# Download and process WHACS offshore waves

Download WHACS bulk and partition spectra, remap them onto metamodel goals, and
write the processed zarr used by dynamic forcing and hindcast reconstruction.

**Prerequisites:** ``04_Select_Goals_by_Hindcasts.ipynb`` (``<Case>_Goals.json``).

## Configuration


In [1]:
import numpy as np
import xarray as xr
from utils.config import get_offshore_wave, load_goals, resolve

whacs_data = xr.open_dataset(
    resolve("inputs/WHACS/processed/Netherlands_WHACS_seapoint.zarr")
).load()
whacs_data = whacs_data.assign({"tp": 1.0 / whacs_data["fp"]})
whacs_data["tp"].attrs = {
    "long_name": "Wave peak period",
    "units": "s",
}
whacs_data

<xarray.Dataset> Size: 1GB
Dimensions:                      (time: 394464, seapoint: 10)
Coordinates:
  * time                         (time) datetime64[ns] 3MB 1979-01-01 ... 202...
  * seapoint                     (seapoint) int32 40B 2323617 ... 2326141
Data variables: (12/42)
    cge                          (time, seapoint) float64 32MB 34.4 15.9 ... 8.9
    dir                          (time, seapoint) float64 32MB 43.6 ... 227.3
    dp                           (time, seapoint) float64 32MB 43.0 ... 262.0
    fp                           (time, seapoint) float64 32MB 0.118 ... 0.129
    hs                           (time, seapoint) float64 32MB 3.01 ... 1.764
    latitude                     (time, seapoint) float32 16MB 52.5 ... 54.0
    ...                           ...
    spr                          (time, seapoint) float64 32MB 27.9 ... 40.3
    t01                          (time, seapoint) float64 32MB 6.36 5.9 ... 4.87
    t02                          (time, seapoint) float64 32MB 5.77 ... 4.39
    uwnd                         (time, seapoint) float64 32MB -5.8 7.4 ... 1.2
    vwnd                         (time, seapoint) float64 32MB -9.6 1.5 ... 11.4
    tp                           (time, seapoint) float64 32MB 8.475 ... 7.752
Attributes: (12/59)
    Conventions:                     CF-1.11 ACDD-1.3
    title:                           Wave Hindcast for the Australian Climate...
    summary:                         Wave Hindcast for the Australian Climate...
    keywords:                        WHACS, ACS, waves, ocean, coast, WWIII, ...
    id:                              BoM-CSIRO-WHACS-001
    naming_authority:                au.csiro
    ...                              ...
    geospatial_lon_max:              179.9375
    geospatial_vertical_positive:    up
    time_coverage_start:             1979-01-01T00:00:00
    time_coverage_end:               1979-01-31T23:00:00
    time_coverage_duration:          P30DT23H0M0S
    time_coverage_resolution:        P0DT1H0M0S

In [2]:
# Offshore node per goal, matched by lon/lat to the goals JSON snap points
# (hindcast.whacs.longitude/latitude). Some hindcasts are gridded rather
# than point-organized, so matching by position is more robust than by id.
goals = load_goals()
seapoint_lon = whacs_data["longitude"].isel(time=-1)
seapoint_lat = whacs_data["latitude"].isel(time=-1)
ordered_seapoints_for_goals = []
for goal in goals.values():
    snap = goal["hindcast"]["whacs"]
    distance = np.hypot(
        seapoint_lon - snap["longitude"], seapoint_lat - snap["latitude"]
    )
    ordered_seapoints_for_goals.append(
        int(whacs_data["seapoint"][int(distance.argmin())])
    )

whacs_data = whacs_data.sel(seapoint=ordered_seapoints_for_goals)[
    [
        "hs",
        "tp",
        "dir",
        "spr",
        "phs0",
        "ptp0",
        "pdir0",
        "pspr0",
        "phs1",
        "ptp1",
        "pdir1",
        "pspr1",
        "phs2",
        "ptp2",
        "pdir2",
        "pspr2",
        "phs3",
        "ptp3",
        "pdir3",
        "pspr3",
        "ppe0",
        "ppe1",
        "ppe2",
        "ppe3",
    ]
].rename({"seapoint": "goal"})
whacs_data["goal"] = [int(k) for k in goals]
whacs_data = whacs_data.assign_coords(
    {"seapoint": ("goal", ordered_seapoints_for_goals)}
)
whacs_data.to_zarr(resolve(get_offshore_wave("WHACS")["data_path"]))

In [ ]:
xr.open_zarr(resolve(get_offshore_wave("WHACS")["data_path"])).load()

<xarray.Dataset> Size: 216B
Dimensions:   (time: 1)
Coordinates:
  * time      (time) datetime64[ns] 8B 1987-01-31T23:00:00
    goal      int64 8B 3
    seapoint  int64 8B 2324234
Data variables: (12/24)
    dir       (time) float64 8B 349.2
    hs        (time) float64 8B 0.658
    pdir0     (time) float64 8B nan
    pdir1     (time) float64 8B 347.5
    pdir2     (time) float64 8B 351.3
    pdir3     (time) float64 8B nan
    ...        ...
    ptp0      (time) float64 8B nan
    ptp1      (time) float64 8B 7.5
    ptp2      (time) float64 8B 10.11
    ptp3      (time) float64 8B nan
    spr       (time) float64 8B 17.0
    tp        (time) float64 8B 10.1
Attributes: (12/59)
    Conventions:                     CF-1.11 ACDD-1.3
    title:                           Wave Hindcast for the Australian Climate...
    summary:                         Wave Hindcast for the Australian Climate...
    keywords:                        WHACS, ACS, waves, ocean, coast, WWIII, ...
    id:                              BoM-CSIRO-WHACS-001
    naming_authority:                au.csiro
    ...                              ...
    geospatial_lon_max:              179.9375
    geospatial_vertical_positive:    up
    time_coverage_start:             1979-01-01T00:00:00
    time_coverage_end:               1979-01-31T23:00:00
    time_coverage_duration:          P30DT23H0M0S
    time_coverage_resolution:        P0DT1H0M0S